In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# Import the CRUD object from Project One
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "pokopiarulez"

# Connect to the database through the CRUD module
db = AnimalShelter(username, password)

# Pull in all records first so the dashboard starts in an unfiltered state
df = pd.DataFrame.from_records(db.read({}))

# MongoDB includes the _id field by default, but Dash tables do not like ObjectId types
# so I am dropping that column before sending the data into the table
df.drop(columns=['_id'], inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)

#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

image_filename = 'Grazioso_Salvare_Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    # Dashboard title, my name for the required unique identifier, and the client logo
    html.Center([
        html.H1('CS-340 Dashboard'),
        html.H4('Aaron Kellan-Fernandez'),
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image.decode()),
            style={'width': '250px'}
        )
    ]),
    
    html.Hr(),

    # Radio buttons used to filter the dogs by rescue category
    html.Div([
        html.Label('Rescue Type Filter'),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'All', 'value': 'all'},
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain or Wilderness', 'value': 'mountain'},
                {'label': 'Disaster or Individual in Danger', 'value': 'disaster'},
            ],
            value='all',
            inline=True
        )
    ]),

    html.Hr(),

    # Interactive data table that starts with all records and then updates from the filter selection
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        sort_action='native',
        filter_action='native',
        row_selectable='single',
        selected_rows=[0],
        style_table={'overflowX': 'auto'},
        style_cell={
            'textAlign': 'left',
            'minWidth': '120px',
            'width': '120px',
            'maxWidth': '180px',
            'overflow': 'hidden',
            'textOverflow': 'ellipsis'
        }
    ),

    html.Br(),
    html.Hr(),

    # This keeps the chart and map side by side on the dashboard
    html.Div(
        className='row',
        style={'display': 'flex'},
        children=[
            html.Div(
                id='graph-id',
                className='col s12 m6',
            ),
            html.Div(
                id='map-id',
                className='col s12 m6',
            )
        ]
    )
])

#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    # Each filter builds a different MongoDB query based on the rescue type requirements
    if filter_type == 'water':
        query = {
            "animal_type": "Dog",
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
            "breed": {"$in": [
                "Labrador Retriever Mix",
                "Chesapeake Bay Retriever",
                "Newfoundland"
            ]}
        }

    elif filter_type == 'mountain':
        query = {
            "animal_type": "Dog",
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
            "breed": {"$in": [
                "German Shepherd",
                "Alaskan Malamute",
                "Old English Sheepdog",
                "Siberian Husky",
                "Rottweiler"
            ]}
        }

    elif filter_type == 'disaster':
        query = {
            "animal_type": "Dog",
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300},
            "breed": {"$in": [
                "Doberman Pinscher",
                "German Shepherd",
                "Golden Retriever",
                "Bloodhound",
                "Rottweiler"
            ]}
        }

    elif filter_type == 'all':
        # Reset returns the full dataset again
        query = {}

    df = pd.DataFrame.from_records(db.read(query))

    # Drop _id again after each filtered read so the table still renders correctly
    if '_id' in df.columns:
        df.drop(columns=['_id'], inplace=True)

    return df.to_dict('records')


@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    # Build the chart from whatever data is currently visible in the data table
    if viewData is None:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    return [
        dcc.Graph(
            figure=px.pie(
                dff,
                names='breed',
                title='Preferred Breeds in Current Filter'
            )
        )
    ]


@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    # When the dashboard first loads, no columns are selected yet
    # so selected_columns will be None and trying to loop through it will crash
    if selected_columns is None:
        return []

    # Highlight whichever column the user selects in the table
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# Update the map based on the selected row in the current table view
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):
    if viewData is None:
        return
    elif index is None:
        return

    dff = pd.DataFrame.from_dict(viewData)

    # Only one row is selectable, so I can safely use the first selected index
    if index is None:
        row = 0
    else:
        row = index[0]

    # Austin TX is centered at [30.75, -97.48]
    return [
        dl.Map(
            style={'width': '1000px', 'height': '500px'},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[dff.iloc[row, 13], dff.iloc[row, 14]],
                    children=[
                        dl.Tooltip(dff.iloc[row, 4]),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(dff.iloc[row, 9])
                        ])
                    ]
                )
            ]
        )
    ]


# Run the app in JupyterLab
app.run_server()

Dash app running on https://bundleimmune-vacuumanswer-3000.codio.io/proxy/8050/
